# 🍁 NOC TEER Classification Pipeline

**Automatically classify job titles → Canada's National Occupational Classification (NOC) TEER codes**

> Built during an 8-month ML Engineering Co-op at **goeasy Ltd.** (Risk team).  
> Processed **4.7 million** loan applicant records in production.

---

## What is TEER?

Canada's [NOC system](https://www.canada.ca/en/immigration-refugees-citizenship/services/immigrate-canada/express-entry/eligibility/find-national-occupation-code.html) assigns every occupation two values:

| Code | Name | Range | Meaning |
|------|------|-------|---------|
| **Hierarchical TEER** | Education level required | 0–5 | 0 = Management, 5 = No formal education needed |
| **Field TEER** | Occupational category | 0–9 | 2 = Science/Tech, 6 = Sales/Service, etc. |

**The problem:** Loan applicants self-report job titles in free text — typos, abbreviations, French titles, gibberish. This pipeline turns that noise into reliable TEER codes.

---

## Pipeline Architecture

```
Raw job titles (messy, bilingual, noisy)
        │
        ▼
┌─────────────────────────┐
│  Step 1: Data Cleaning  │  Hybrid French detection · Gibberish flagging · Normalization
└────────────┬────────────┘
             │
             ▼
┌─────────────────────────┐
│  Step 2: NOC Matching   │  16 override rules · Fuzzy match vs 27K titles · Penalty scoring
└────────────┬────────────┘
             │
             ▼
┌─────────────────────────┐
│  Step 3: ML Training    │  TF-IDF + RandomForest (500 trees) + Isotonic Calibration
└────────────┬────────────┘
             │
             ▼
     TEER Predictions + Confidence Scores
```

---

## Results

| Dataset | Hierarchical TEER Acc. | Field TEER Acc. | Avg Confidence |
|---------|------------------------|-----------------|----------------|
| High-confidence (score ≥ 85) | **95.08%** | **95.61%** | ~92% |
| All predictions (score ≥ 75) | **77.56%** | **78.97%** | ~84% |


---
## 📦 Setup & Installation

Run this cell first to install all dependencies. Works on **Google Colab** and any standard Jupyter environment.


In [ ]:
# Install required packages
!pip install rapidfuzz langdetect scikit-learn pandas numpy matplotlib -q

import pandas as pd
import numpy as np
import re
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

print("✅ All packages installed successfully")
print(f"   Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


---
## Step 1 — Data Cleaning

Three sub-tasks:
1. **Normalize** text (lowercase, strip special characters)
2. **Detect language** — filter out French job titles using a hybrid approach
3. **Flag gibberish** — catch placeholders, non-job titles (student, retired, etc.), numeric strings

### Why hybrid French detection?
`langdetect` alone is unreliable on very short strings (e.g. it misclassifies "préposé" as English). 
We combine:
- Unicode character matching for unambiguous French characters (é, à, ç, ê...)
- `langdetect` as a secondary signal
- A curated allowlist of unambiguous French function words (au, aux, des, les...)

This gives **precision** — we only remove a title if we're confident it's French.


In [ ]:
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

DetectorFactory.seed = 0  # reproducibility

def normalize_title(title: str) -> str:
    """Lowercase and strip non-alphanumeric characters (keep hyphens)."""
    if not isinstance(title, str):
        return ""
    return re.sub(r"[^a-zA-Z0-9\s-]", "", title).strip().lower()

# Unambiguous French Unicode characters
FRENCH_CHAR_RE = re.compile(
    r"[\u00e9\u00e0\u00e7\u00ea\u00ee\u00f4\u00fb\u00eb\u00ef\u00fc\u0153]"
)

# Unambiguous French function words (rare in English job titles)
FRENCH_WORDS = {
    'au', 'aux', 'avec', 'ces', 'dans', 'des', 'elle', 'entre', 'ils',
    'les', 'leur', 'leurs', 'mes', 'mon', 'nos', 'notre', 'pour',
    'que', 'qui', 'sans', 'ses', 'son', 'sur', 'tes', 'ton', 'vos', 'votre'
}

def detect_language(title: str) -> str:
    """
    Hybrid language detector. Returns 'fr', 'en', or 'unknown'.
    Only labels French if there's strong evidence — errs on keeping English.
    """
    if not title or not isinstance(title, str) or title.strip() == "":
        return "unknown"
    
    # Strong signal: unambiguous French Unicode character
    if FRENCH_CHAR_RE.search(title.lower()):
        return 'fr'
    
    # Secondary: langdetect vote
    try:
        langdetect_vote = detect(title.lower())
    except LangDetectException:
        langdetect_vote = 'unknown'
    
    # Only accept langdetect's 'fr' if confirmed by a French function word
    if langdetect_vote == 'fr':
        title_words = set(title.lower().split())
        if not title_words.isdisjoint(FRENCH_WORDS):
            return 'fr'
    
    return 'en'

# ── Gibberish / non-job-title keywords ────────────────────────────────────
GIBBERISH_SET = {
    "ask", "tba", "tbd", "nan", "xyz", "add", "tbh", "abc", "none",
    "student", "retired", "unemployed", "disability", "n/a",
    "pension", "pensioner", "cpp", "odsp", "self-employed", "self employed", "se"
}

def is_gibberish(title: str) -> bool:
    """Flag non-job titles: empty, too short, numeric-heavy, or known placeholders."""
    if not title or len(title.strip()) < 3:
        return True
    if title in GIBBERISH_SET:
        return True
    alpha_only = re.sub(r"[^a-zA-Z]", "", title)
    if not alpha_only:  # numbers-only string
        return True
    digit_count = len(re.sub(r"[^0-9]", "", title))
    if digit_count / len(title) > 0.7:  # >70% digits
        return True
    return False

print("✅ Cleaning functions defined")


In [ ]:
# ── Demo on sample job titles ──────────────────────────────────────────────
sample_titles = [
    "Software Engineer",
    "Ingénieur logiciel",           # French — should be filtered
    "préposé aux bénéficiaires",    # French — should be filtered
    "cashier",
    "student",                       # Non-job — gibberish
    "tbd",                           # Placeholder — gibberish
    "1234",                          # Numbers only — gibberish
    "RN",
    "sr",                            # Too short
    "Machine Learning Engineer",
    "caissière",                     # French — should be filtered
    "Nurse Practitioner",
    "retired",                       # Non-job
]

results = []
for title in sample_titles:
    cleaned = normalize_title(title)
    lang = detect_language(title)
    gibberish = is_gibberish(cleaned)
    keep = lang == 'en' and not gibberish
    results.append({
        "Original": title,
        "Cleaned": cleaned,
        "Language": lang,
        "Gibberish": gibberish,
        "✅ Keep": keep
    })

df_demo = pd.DataFrame(results)
print(df_demo.to_string(index=False))


---
## Step 2 — NOC Code Matching

Two-stage matching:

### Stage A — Override Rules
16 hand-crafted mappings for extremely common one-word job titles (`cashier`, `driver`, `manager`...).  
These get a score of 100 and bypass fuzzy matching entirely.

### Stage B — Fuzzy Matching
For everything else: match against ~27,000 official NOC titles using `rapidfuzz`.

**Scorer:** `token_set_ratio` — handles word-order differences well  
**Penalty:** −20 points per unmatched token in the NOC title  
**Threshold:** Final score ≥ 75 to accept a match

**Key optimization:** Deduplicate titles first, fuzzy match unique titles only, then broadcast results back. With 4.7M records but far fewer unique titles, this cuts runtime dramatically.


In [ ]:
# ── Override rules (common single-word titles) ───────────────────────────
OVERRIDES = {
    "labor":        ("95109", "Other labourers in processing, manufacturing and utilities"),
    "labour":       ("95109", "Other labourers in processing, manufacturing and utilities"),
    "labourer":     ("95109", "Other labourers in processing, manufacturing and utilities"),
    "worker":       ("95109", "Other labourers in processing, manufacturing and utilities"),
    "sales":        ("64100", "Retail salespersons and visual merchandisers"),
    "associate":    ("64100", "Retail salespersons and visual merchandisers"),
    "cashier":      ("65100", "Cashiers"),
    "manager":      ("00018", "Senior managers - public and private sector"),
    "owner":        ("00018", "Senior managers - public and private sector"),
    "president":    ("00012", "Senior managers - financial, communications and other business services"),
    "admin":        ("13110", "Administrative assistants"),
    "receptionist": ("14101", "Receptionist"),
    "driver":       ("73300", "Transport truck drivers"),
    "staff":        ("14100", "General office support clerks"),
    "teacher":      ("41221", "Elementary school and kindergarten teachers"),
    "delivery":     ("75201", "Delivery service drivers and door-to-door distributors"),
}

print(f"✅ {len(OVERRIDES)} override rules loaded")


In [ ]:
from rapidfuzz import fuzz, process

# ── Sample NOC titles (subset of official 27K list) ───────────────────────
# In production this loads from: file:/Workspace/Shared/NOC Project/noc_titles.csv
# Here we use a representative sample of public NOC data.
SAMPLE_NOC_TITLES = [
    ("00011", "Legislators"),
    ("00012", "Senior government managers and officials"),
    ("00013", "Senior managers - financial, communications and other business services"),
    ("00014", "Senior managers - health, education, social and community services"),
    ("00015", "Senior managers - trade, broadcasting and other services"),
    ("00016", "Senior managers - construction, transportation, production and utilities"),
    ("00018", "Senior managers - public and private sector"),
    ("11100", "Financial auditors and accountants"),
    ("11101", "Financial analysts and investment advisors"),
    ("11102", "Securities agents, investment dealers and brokers"),
    ("12100", "Administrative services managers"),
    ("12101", "Human resources managers"),
    ("12200", "Procurement officers"),
    ("13100", "Administrative officers"),
    ("13110", "Administrative assistants"),
    ("14100", "General office support workers"),
    ("14101", "Receptionists"),
    ("20011", "Architects"),
    ("20012", "Landscape architects"),
    ("21100", "Physicists and astronomers"),
    ("21110", "Chemists"),
    ("21120", "Geoscientists and oceanographers"),
    ("21200", "Civil engineers"),
    ("21210", "Mechanical engineers"),
    ("21220", "Electrical and electronics engineers"),
    ("21230", "Chemical engineers"),
    ("21232", "Mining engineers"),
    ("21300", "Computer engineers"),
    ("21310", "Computer systems analysts"),
    ("21320", "Information systems managers"),
    ("30010", "Physicians"),
    ("30020", "Dentists"),
    ("31100", "Pharmacists"),
    ("31101", "Dietitians and nutritionists"),
    ("31102", "Audiologists and speech-language pathologists"),
    ("31110", "Physiotherapists"),
    ("31120", "Occupational therapists"),
    ("31200", "Registered nurses and registered psychiatric nurses"),
    ("32100", "Licensed practical nurses"),
    ("32101", "Paramedical occupations"),
    ("40010", "University professors and lecturers"),
    ("40020", "Post-secondary teaching and research assistants"),
    ("41200", "Secondary school teachers"),
    ("41210", "Elementary school and kindergarten teachers"),
    ("41221", "Early childhood educators and assistants"),
    ("42100", "Social workers"),
    ("42201", "Police officers"),
    ("42202", "Firefighters"),
    ("43100", "Elementary and secondary school teacher assistants"),
    ("50010", "Librarians"),
    ("51100", "Authors and writers"),
    ("51110", "Editors"),
    ("51120", "Journalists"),
    ("52100", "Graphic designers and illustrators"),
    ("52110", "Interior designers and interior decorators"),
    ("53100", "Athletes"),
    ("53200", "Coaches"),
    ("60010", "Retail and wholesale trade managers"),
    ("62010", "Chefs"),
    ("62020", "Cooks"),
    ("63100", "Butchers"),
    ("64100", "Retail salespersons and visual merchandisers"),
    ("64101", "Sales and account representatives"),
    ("65100", "Cashiers"),
    ("65101", "Service station attendants"),
    ("65200", "Food counter attendants and kitchen helpers"),
    ("65201", "Food service supervisors"),
    ("65210", "Bartenders"),
    ("65211", "Food and beverage servers"),
    ("65310", "Security guards and related security service occupations"),
    ("65311", "Correctional service officers"),
    ("65320", "Janitors and building superintendents"),
    ("65321", "Cleaners"),
    ("70010", "Construction managers"),
    ("70012", "Home building and renovation managers"),
    ("72010", "Electricians and power system electricians"),
    ("72011", "Industrial electricians"),
    ("72020", "Pipefitters and sprinkler system installers"),
    ("72100", "Carpenters"),
    ("72101", "Cabinetmakers"),
    ("72200", "Welders and related machine operators"),
    ("73100", "Aircraft mechanics and inspectors"),
    ("73200", "Motor vehicle mechanics and repairers"),
    ("73300", "Transport truck drivers"),
    ("73301", "Bus drivers and subway and other transit operators"),
    ("73400", "Longshore workers"),
    ("75100", "Residential and commercial installers and servicers"),
    ("75201", "Delivery service drivers and door-to-door distributors"),
    ("80010", "Managers in agriculture"),
    ("82010", "Agricultural service contractors and farm supervisors"),
    ("82020", "Farmers and farm managers"),
    ("85100", "Harvesting labourers"),
    ("85110", "Nursery and greenhouse workers"),
    ("90010", "Manufacturing managers"),
    ("92010", "Central control and process operators in manufacturing"),
    ("94100", "Machine operators in pulp and paper production"),
    ("95100", "Labourers in mining and quarrying"),
    ("95109", "Other labourers in processing, manufacturing and utilities"),
]

noc_df = pd.DataFrame(SAMPLE_NOC_TITLES, columns=["code", "title"])
noc_df["clean_title"] = noc_df["title"].str.lower().str.replace(r"[^a-z0-9\s-]", "", regex=True).str.strip()

noc_choices = noc_df["clean_title"].tolist()
tokenized_noc = [set(t.split()) for t in noc_choices]

print(f"✅ {len(noc_df)} NOC titles loaded (sample; production uses 27K)")
print(noc_df.head(5).to_string(index=False))


In [ ]:
def match_title(title: str, threshold: float = 75.0):
    """
    Match a single cleaned job title to the best NOC code.
    Returns (title, noc_code, noc_title, score, method) or None if below threshold.
    """
    if not title:
        return None
    
    # Stage A: override rules
    if title in OVERRIDES:
        code, noc_title = OVERRIDES[title]
        return (title, code, noc_title, 100.0, "Override")
    
    # Stage B: fuzzy match
    match = process.extractOne(title, noc_choices, scorer=fuzz.token_set_ratio)
    if not match:
        return None
    
    best_str, score, idx = match
    
    # Penalty: -20 pts per token in the NOC title that doesn't appear in the query
    penalty = len(tokenized_noc[idx].difference(set(title.split()))) * 20.0
    final_score = max(0, score - penalty)
    
    if final_score >= threshold:
        matched_row = noc_df[noc_df["clean_title"] == best_str].iloc[0]
        return (title, matched_row["code"], matched_row["title"], float(final_score), "Fuzzy")
    
    return None

# ── Test on a batch of sample titles ─────────────────────────────────────
test_titles = [
    "software engineer",
    "machine learning engineer",
    "registered nurse",
    "cashier",
    "truck driver",
    "accountant",
    "elementary school teacher",
    "security guard",
    "general labourer",
    "data scientist",
    "marketing specialist",
    "construction worker",
    "pharmacist",
    "graphic designer",
    "real estate agent",
    "hvac technician",
    "sous chef",
    "retail associate",
    "civil engineer",
    "social worker",
]

print(f"{'Job Title':<30} {'NOC Code':<10} {'Matched Title':<45} {'Score':>6} {'Method'}")
print("-" * 105)
for t in test_titles:
    result = match_title(t)
    if result:
        title, code, noc_title, score, method = result
        print(f"{t:<30} {code:<10} {noc_title[:44]:<45} {score:>6.1f} {method}")
    else:
        print(f"{t:<30} {'—':<10} {'No match above threshold':<45} {'—':>6}")


In [ ]:
# ── Parallel matching demo (ThreadPoolExecutor) ───────────────────────────
# In production: 4 threads, batches of 5000, deduplicated titles
# This demos the same pattern on our test set

def parallel_match(titles, n_workers=4):
    """Match a list of unique titles in parallel."""
    results = []
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        batch_results = list(executor.map(match_title, titles))
    results = [r for r in batch_results if r is not None]
    elapsed = time.time() - t0
    rate = len(titles) / elapsed if elapsed > 0 else 0
    print(f"   Matched {len(results)}/{len(titles)} titles in {elapsed:.2f}s ({rate:.0f} titles/sec)")
    return results

print("Running parallel fuzzy match...")
matched = parallel_match(test_titles)

df_matched = pd.DataFrame(matched, columns=["cleaned_title", "noc_code", "noc_title", "score", "method"])
print(f"\n{len(df_matched)} matches found above threshold")
print(df_matched[["cleaned_title", "noc_code", "score", "method"]].to_string(index=False))


---
## Step 3 — Feature Engineering

The ML model uses 7 features derived from each record:

| Feature | Type | Description |
|---------|------|-------------|
| `cleaned_emp_position` | TF-IDF (15K, 1–3 ngrams) | The job title itself |
| `ApplicantEmployersName` | TF-IDF (5K, 1–2 ngrams) | Employer name as context |
| `ApplicantCity` | TF-IDF (2K, 1–2 ngrams) | City as geographic signal |
| `yearly_salary` | Numeric | Annual salary (monthly × 12) |
| `score` | Numeric | Fuzzy match confidence |
| `title_word_count` | Numeric | Number of words in title |
| `title_char_length` | Numeric | Character length of title |

**Why employer name?** "Engineer at CN Rail" vs "Engineer at RBC" → very different NOC codes.  
**Why city?** Certain occupations cluster geographically (e.g. oil-patch trades in Fort McMurray).


In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add derived features. Must be identical at training and prediction time.
    """
    df = df.copy()
    df["cleaned_emp_position"]    = df["cleaned_emp_position"].fillna("unknown").astype(str)
    df["ApplicantEmployersName"]  = df["ApplicantEmployersName"].fillna("unknown").astype(str)
    df["ApplicantCity"]           = df["ApplicantCity"].fillna("unknown").astype(str)
    df["yearly_salary"]           = df["yearly_salary"].fillna(0.0)
    df["score"]                   = df["score"].fillna(0.0)
    df["title_word_count"]        = df["cleaned_emp_position"].str.split().str.len().fillna(0).astype(float)
    df["title_char_length"]       = df["cleaned_emp_position"].str.len().fillna(0).astype(float)
    return df

FEATURE_COLS = [
    "cleaned_emp_position",
    "ApplicantEmployersName",
    "ApplicantCity",
    "yearly_salary",
    "score",
    "title_word_count",
    "title_char_length"
]

# ── Build a synthetic training set from our matched results ───────────────
# (In production this is the full 4.7M applicant dataset)
synthetic_records = []
employer_map = {
    "software engineer": ("Tech Corp", "Toronto", 95000),
    "machine learning engineer": ("AI Startup", "Toronto", 110000),
    "registered nurse": ("Toronto General Hospital", "Toronto", 78000),
    "cashier": ("Loblaw Companies", "Brampton", 38000),
    "truck driver": ("TransForce Inc", "Mississauga", 62000),
    "accountant": ("Deloitte Canada", "Toronto", 72000),
    "elementary school teacher": ("TDSB", "Toronto", 68000),
    "security guard": ("Securitas Canada", "Mississauga", 42000),
    "general labourer": ("ABC Construction", "Hamilton", 45000),
    "graphic designer": ("Creative Agency", "Toronto", 58000),
    "civil engineer": ("AECOM Canada", "Toronto", 85000),
    "social worker": ("CAS Toronto", "Toronto", 62000),
    "pharmacist": ("Shoppers Drug Mart", "Toronto", 95000),
    "sous chef": ("Fairmont Hotels", "Toronto", 55000),
    "retail associate": ("Hudson Bay", "Toronto", 36000),
}

for r in matched:
    title, code, noc_title, score, method = r
    employer, city, salary = employer_map.get(title, ("Unknown Company", "Toronto", 50000))
    # Extract TEER digits from NOC code
    if len(code) >= 2:
        field_teer = int(code[0])
        hier_teer  = int(code[1])
        synthetic_records.append({
            "cleaned_emp_position": title,
            "ApplicantEmployersName": employer,
            "ApplicantCity": city,
            "yearly_salary": float(salary),
            "score": score,
            "matched_code": code,
            "Field_NOC_TEER": field_teer,
            "Hierarchical_NOC_TEER": hier_teer,
        })

train_df = pd.DataFrame(synthetic_records)
train_df = engineer_features(train_df)

print(f"✅ Synthetic training set: {len(train_df)} records")
print("\nFeature preview:")
print(train_df[FEATURE_COLS + ["Field_NOC_TEER", "Hierarchical_NOC_TEER"]].to_string(index=False))


---
## Step 4 — Model Training

### Architecture

```
ColumnTransformer
├── TfidfVectorizer(job_title,    max_features=15000, ngram_range=(1,3), sublinear_tf=True)
├── TfidfVectorizer(employer,     max_features=5000,  ngram_range=(1,2))
├── TfidfVectorizer(city,         max_features=2000,  ngram_range=(1,2))
└── StandardScaler([salary, score, word_count, char_length])
        │
        ▼
RandomForestClassifier(n_estimators=500, max_depth=30, class_weight="balanced")
        │
        ▼
CalibratedClassifierCV(method="isotonic")  ← key design decision
```

### Why calibration matters

Raw Random Forest confidence scores are **poorly calibrated** — the model says "95% confident" when it's actually right only 70% of the time. Isotonic regression calibration aligns confidence to real accuracy. This is critical when routing uncertain predictions to human review.

### Data split: 60 / 20 / 20
- **60% Train** → fit the Random Forest
- **20% Calibration** → fit isotonic calibration layer (must be held out from RF training)
- **20% Test** → final evaluation (never seen during training or calibration)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report

def build_pipeline():
    """
    Build the full preprocessing + Random Forest pipeline.
    Identical for both Hierarchical and Field TEER models.
    """
    preprocessor = ColumnTransformer([
        ("job",     TfidfVectorizer(max_features=15000, ngram_range=(1,3),
                                    stop_words="english", sublinear_tf=True),
                    "cleaned_emp_position"),
        ("employer",TfidfVectorizer(max_features=5000,  ngram_range=(1,2),
                                    stop_words="english"),
                    "ApplicantEmployersName"),
        ("city",    TfidfVectorizer(max_features=2000,  ngram_range=(1,2),
                                    stop_words="english"),
                    "ApplicantCity"),
        ("numeric", StandardScaler(),
                    ["yearly_salary", "score", "title_word_count", "title_char_length"])
    ])
    rf = RandomForestClassifier(
        n_estimators=500,
        max_depth=30,
        min_samples_leaf=3,
        min_samples_split=5,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )
    return SkPipeline([("prep", preprocessor), ("rf", rf)])

print("✅ Pipeline builder defined")
print("   Features: job title TF-IDF (15K) + employer (5K) + city (2K) + 4 numeric")
print("   Model: RandomForest(500 trees, depth=30, balanced classes)")


In [ ]:
# ── Note on sample size ───────────────────────────────────────────────────
# The synthetic dataset here is small (for demo purposes).
# In production, models were trained on millions of high-confidence records.
# With a tiny demo set we'll use a simplified split; the architecture is identical.

X = train_df[FEATURE_COLS]
y_hier  = train_df["Hierarchical_NOC_TEER"]
y_field = train_df["Field_NOC_TEER"]

print(f"Training set size: {len(X)} records")
print(f"Hierarchical TEER classes: {sorted(y_hier.unique())}")
print(f"Field TEER classes:        {sorted(y_field.unique())}")

# For demo: use 60/40 split (calibration built into CV with small data)
X_train, X_test, yh_train, yh_test, yf_train, yf_test = train_test_split(
    X, y_hier, y_field, test_size=0.3, random_state=42
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")


In [ ]:
# ── Train Hierarchical TEER model ────────────────────────────────────────
print("Training Hierarchical TEER model...")
t0 = time.time()

h_base = build_pipeline()
h_base.fit(X_train, yh_train)

# Calibration via cross-validation (cv=3 works with small datasets)
h_calibrated = CalibratedClassifierCV(h_base, cv="prefit", method="isotonic")
h_calibrated.fit(X_train, yh_train)  # refit calibration on training set (demo only)

h_preds = h_calibrated.predict(X_test)
h_conf  = np.max(h_calibrated.predict_proba(X_test), axis=1)
h_acc   = accuracy_score(yh_test, h_preds)

print(f"   Done in {time.time()-t0:.1f}s")
print(f"   Accuracy:        {h_acc:.2%}")
print(f"   Avg Confidence:  {h_conf.mean():.2%}")


In [ ]:
# ── Train Field TEER model ───────────────────────────────────────────────
print("Training Field TEER model...")
t0 = time.time()

f_base = build_pipeline()
f_base.fit(X_train, yf_train)

f_calibrated = CalibratedClassifierCV(f_base, cv="prefit", method="isotonic")
f_calibrated.fit(X_train, yf_train)

f_preds = f_calibrated.predict(X_test)
f_conf  = np.max(f_calibrated.predict_proba(X_test), axis=1)
f_acc   = accuracy_score(yf_test, f_preds)

print(f"   Done in {time.time()-t0:.1f}s")
print(f"   Accuracy:        {f_acc:.2%}")
print(f"   Avg Confidence:  {f_conf.mean():.2%}")

print(f"\n{'='*60}")
print("MODEL SUMMARY")
print(f"{'='*60}")
print(f"  Hierarchical TEER: {h_acc:.2%} accuracy | {h_conf.mean():.2%} avg confidence")
print(f"  Field TEER:        {f_acc:.2%} accuracy | {f_conf.mean():.2%} avg confidence")
print(f"{'='*60}")


---
## Step 5 — Prediction & Results Visualization

Run inference on new job titles and visualize the distribution across TEER categories.


In [ ]:
# ── TEER label maps ──────────────────────────────────────────────────────
HIER_LABELS = {
    0: 'Management',
    1: 'University Degree',
    2: 'College / Apprenticeship (2yr+)',
    3: 'Apprenticeship / On-the-Job',
    4: 'High School Diploma',
    5: 'No Formal Education'
}

FIELD_LABELS = {
    0: 'Management',
    1: 'Business & Finance',
    2: 'Science & Technology',
    3: 'Health',
    4: 'Education & Social Services',
    5: 'Arts & Culture',
    6: 'Sales & Service',
    7: 'Trades & Transport',
    8: 'Agriculture & Resources',
    9: 'Manufacturing & Utilities'
}

# ── Predict on new titles ─────────────────────────────────────────────────
new_titles = [
    ("data analyst", "RBC", "Toronto", 75000),
    ("warehouse worker", "Amazon Canada", "Mississauga", 42000),
    ("pediatric nurse", "Sick Kids Hospital", "Toronto", 82000),
    ("restaurant manager", "Tim Hortons", "Ottawa", 55000),
    ("electrician", "Hydro One", "Toronto", 78000),
    ("customer service rep", "Bell Canada", "Montreal", 45000),
    ("financial advisor", "TD Wealth", "Toronto", 90000),
    ("early childhood educator", "YMCA", "Hamilton", 40000),
]

# Clean and match titles first
matched_new = []
for title, employer, city, salary in new_titles:
    cleaned = normalize_title(title)
    lang = detect_language(title)
    gibberish = is_gibberish(cleaned)
    if lang == 'en' and not gibberish:
        result = match_title(cleaned)
        if result:
            _, code, noc_title, score, method = result
            matched_new.append({
                "cleaned_emp_position": cleaned,
                "ApplicantEmployersName": employer,
                "ApplicantCity": city,
                "yearly_salary": float(salary),
                "score": score,
                "matched_code": code,
            })

new_df = pd.DataFrame(matched_new)
if not new_df.empty:
    new_df = engineer_features(new_df)
    X_new = new_df[FEATURE_COLS]
    
    h_new = h_calibrated.predict(X_new)
    h_new_conf = np.max(h_calibrated.predict_proba(X_new), axis=1)
    f_new = f_calibrated.predict(X_new)
    f_new_conf = np.max(f_calibrated.predict_proba(X_new), axis=1)
    
    print(f"{'Job Title':<28} {'Hier TEER':<30} {'Conf':>5}  {'Field TEER':<25} {'Conf':>5}")
    print("-" * 100)
    for i, row in new_df.iterrows():
        h_label = HIER_LABELS.get(h_new[i], str(h_new[i]))
        f_label = FIELD_LABELS.get(f_new[i], str(f_new[i]))
        print(f"{row['cleaned_emp_position']:<28} {h_label:<30} {h_new_conf[i]:>5.0%}  {f_label:<25} {f_new_conf[i]:>5.0%}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'figure.facecolor': 'white', 'axes.facecolor': '#FAFAFA',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.spines.left': False, 'axes.grid': True,
    'grid.alpha': 0.15, 'grid.axis': 'x'
})

def conf_color(c):
    if c >= 90: return '#2E7D32'
    elif c >= 70: return '#E65100'
    else: return '#C62828'

def conf_bg(c):
    if c >= 90: return '#E8F5E9'
    elif c >= 70: return '#FFF3E0'
    else: return '#FFEBEE'

def plot_teer_breakdown(preds, confs, labels, title, bar_color):
    """Horizontal bar chart: customer count per TEER category + confidence badges."""
    unique_vals = sorted(np.unique(preds))
    cats      = [labels.get(v, str(v)) for v in unique_vals]
    counts    = [int(np.sum(preds == v)) for v in unique_vals]
    avg_confs = [float(confs[preds == v].mean() * 100) for v in unique_vals]
    total     = sum(counts)

    # Sort largest first
    order     = np.argsort(counts)[::-1]
    cats      = [cats[i] for i in order]
    counts    = [counts[i] for i in order]
    avg_confs = [avg_confs[i] for i in order]

    fig, ax = plt.subplots(figsize=(14, max(len(cats) * 0.9 + 1.5, 5)))
    y    = np.arange(len(cats))
    bars = ax.barh(y, counts, height=0.58, color=bar_color, alpha=0.85,
                   edgecolor='white', linewidth=1.2)

    max_count = max(counts) if counts else 1
    for i, bar in enumerate(bars):
        pct = counts[i] / total * 100
        ax.text(bar.get_width() + max_count * 0.02,
                bar.get_y() + bar.get_height() / 2,
                f'{counts[i]:,}  ({pct:.1f}%)',
                va='center', ha='left', fontsize=11,
                fontweight='bold', color='#333333')
        cc = avg_confs[i]
        ax.text(bar.get_width() + max_count * 0.25,
                bar.get_y() + bar.get_height() / 2,
                f'  {cc:.0f}% confidence  ',
                va='center', ha='left', fontsize=9.5, fontweight='bold',
                color=conf_color(cc),
                bbox=dict(boxstyle='round,pad=0.35',
                          facecolor=conf_bg(cc),
                          edgecolor=conf_color(cc),
                          alpha=0.9, linewidth=1.3))

    ax.set_yticks(y)
    ax.set_yticklabels(cats, fontsize=12, fontweight='medium')
    ax.invert_yaxis()
    ax.set_xlabel('Number of Records', fontsize=11, labelpad=8)
    ax.set_title(title, fontsize=16, fontweight='bold', pad=16, loc='left')
    ax.set_xlim(0, max_count * 1.6)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
    ax.tick_params(axis='y', length=0)

    overall_conf = float(np.mean(avg_confs)) if avg_confs else 0
    ax.text(0.99, -0.08,
            f'Total: {total:,} records   |   Avg Confidence: {overall_conf:.0f}%',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=10, color='#777777', style='italic')

    legend_items = [
        mpatches.Patch(facecolor='#E8F5E9', edgecolor='#2E7D32', label='High (≥90%)'),
        mpatches.Patch(facecolor='#FFF3E0', edgecolor='#E65100', label='Medium (70–90%)'),
        mpatches.Patch(facecolor='#FFEBEE', edgecolor='#C62828', label='Low (<70%)'),
    ]
    ax.legend(handles=legend_items, loc='lower right', fontsize=9,
              framealpha=0.9, title='Confidence', title_fontsize=9)

    fig.tight_layout()
    plt.show()

# Use training set predictions for the charts (larger sample than new_titles)
# In production these run on 4.7M records
all_h_preds = h_calibrated.predict(X)
all_h_conf  = np.max(h_calibrated.predict_proba(X), axis=1)
all_f_preds = f_calibrated.predict(X)
all_f_conf  = np.max(f_calibrated.predict_proba(X), axis=1)

plot_teer_breakdown(all_h_preds, all_h_conf, HIER_LABELS,
                    'Records by Education Level (Hierarchical TEER)', '#1565C0')
plot_teer_breakdown(all_f_preds, all_f_conf, FIELD_LABELS,
                    'Records by Job Category (Field TEER)', '#D84315')


In [ ]:
# ── Confidence distribution plot ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, confs, title, color in [
    (axes[0], all_h_conf, "Hierarchical TEER Confidence", "#1565C0"),
    (axes[1], all_f_conf, "Field TEER Confidence",        "#D84315"),
]:
    ax.hist(confs * 100, bins=20, color=color, alpha=0.78, edgecolor='white', linewidth=0.8)
    ax.axvline(confs.mean() * 100, color='#333', linestyle='--', linewidth=1.5,
               label=f'Mean: {confs.mean()*100:.1f}%')
    ax.axvline(75, color='#E65100', linestyle=':', linewidth=1.3,
               label='Review threshold (75%)')
    ax.set_xlabel('Confidence Score (%)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold', loc='left')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Model Confidence Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
## Summary & Key Takeaways

### What this pipeline does
Takes a messy, bilingual free-text job title and outputs:
- A standardized NOC code
- A Hierarchical TEER (education level required)
- A Field TEER (occupational category)  
- A calibrated confidence score

### Key design decisions

| Decision | Why |
|----------|-----|
| Hybrid French detection | `langdetect` alone is unreliable on short strings |
| Deduplicate before fuzzy matching | 4.7M records → far fewer unique titles → massive speedup |
| Fuzzy match before ML | Creates interpretable, high-quality training labels |
| Isotonic calibration | Raw RF confidence is overconfident; calibration makes scores meaningful |
| Separate models for Hierarchical & Field | Different class structures, different error profiles |

### Production results
- **95% accuracy** on high-confidence matches (score ≥ 85)
- **78% accuracy** across all applicant records (score ≥ 75)
- Average confidence ~84–92% depending on match quality

---
*Built by Yash Patel · ML Engineering Co-op @ goeasy Ltd.*  
*GitHub: [github.com/Yash3842](https://github.com/Yash3842) · LinkedIn: [yash-patel-35449a226](https://www.linkedin.com/in/yash-patel-35449a226/)*
